# 03 — Manual Review & Spot-Check (LOATO-2A-03)

Interactive workflow for human spot-checking of LLM-assigned labels and applying manual overrides.

In [ ]:
import pandas as pd

from loato_bench.utils.config import DATA_DIR

# Load labeled dataset
df = pd.read_parquet(DATA_DIR / "processed" / "labeled_v1.parquet")
print(f"Loaded {len(df):,} samples")
print("\nLabel source breakdown:")
print(df["label_source"].value_counts())
print("\nAttack category breakdown (injection only):")
print(df[df["label"] == 1]["attack_category"].value_counts())

In [ ]:
from loato_bench.data.review import export_spot_check_samples

# Export stratified spot-check samples (50 per category)
spot_check = export_spot_check_samples(df, n_per_category=50, seed=42)
print(f"Spot-check samples: {len(spot_check)}")
print("\nPer category:")
print(spot_check["attack_category"].value_counts())

# Save to CSV
out_dir = DATA_DIR / "review"
out_dir.mkdir(parents=True, exist_ok=True)
spot_check.to_csv(out_dir / "spot_check.csv", index=False)
spot_check.head()

## Boundary Rules Reference

When reviewing samples, use these boundary rules from the taxonomy spec:

| ID | Category | Key signal | NOT this category |
|----|----------|-----------|-------------------|
| C1 | Instruction Override | "ignore previous", "disregard rules" | Persona adoption → C2 |
| C2 | Jailbreak / Roleplay | "pretend you are", "act as" | No persona → C1 |
| C3 | Obfuscation / Encoding | base64, ROT13, leetspeak | Plaintext override → C1 |
| C4 | Information Extraction | "reveal system prompt", "show password" | Override to do something new → C1 |
| C5 | Social Engineering | urgency, authority claims, guilt | Persona-based → C2 |
| C6 | Context Manipulation | hidden text, tool output injection | Direct user override → C1 |
| C7 | Other / Multi-Strategy | Multiple strategies, no clear primary | One strategy dominates → that category |

In [ ]:
from loato_bench.data.review import export_uncertain_pool

# Export uncertain pool for full review
uncertain = export_uncertain_pool(df)
print(f"Uncertain samples: {len(uncertain)}")
uncertain.to_csv(out_dir / "uncertain_pool.csv", index=False)
uncertain.head()

## Override Template

After reviewing, create `manual_overrides.csv` with columns:
- `sample_hash` — from the exported CSV
- `correct_category` — slug (e.g. `instruction_override`) or C-ID (e.g. `C1`)

Only include rows where you disagree with the LLM label. Leave others out.

In [ ]:
import json

from loato_bench.data.review import (
    apply_manual_overrides,
    compute_error_rates,
    generate_coverage_report_v2,
    load_manual_overrides,
)

# After human review, apply overrides:
overrides_path = out_dir / "manual_overrides.csv"
if overrides_path.exists():
    overrides = load_manual_overrides(overrides_path)
    df_updated = apply_manual_overrides(df, overrides)

    # Compute error rates from spot-check
    sc = pd.read_csv(out_dir / "spot_check.csv")
    errors = compute_error_rates(sc)
    print("Error rates:", json.dumps(errors, indent=2))

    # Generate coverage report
    report = generate_coverage_report_v2(df_updated)
    print("\nCoverage report:", json.dumps(report, indent=2))
else:
    print(f"No overrides file found at {overrides_path}.")
    print("Create it after reviewing the spot-check and uncertain CSVs.")